In [29]:
import pandas as pd
import numpy as np

from joblib import dump, load

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix

### Classification of Consumer Complaints

>The Consumer Financial Protection Bureau publishes the Consumer Complaint Database, a collection of complaints about consumer financial products and services that were sent to companies for response. Complaints are published after the company responds, confirming a commercial relationship with the consumer, or after 15 days, whichever comes first. 
>
>You have been provided with a dataset of over 350,000 such complaints for 5 common issue types. Your goal is to train a text classification model to identify the issue type based on the consumer complaint narrative. The data can be downloaded from https://drive.google.com/file/d/1Hz1gnCCr-SDGjnKgcPbg7Nd3NztOLdxw/view?usp=share_link 

In [2]:
complaints = pd.read_csv("../data/complaints.csv")

In [3]:
complaints.head()

,Consumer complaint narrative,Issue
0,My name is XXXX XXXX this complaint is not mad...,Incorrect information on your report
1,I searched on XXXX for XXXXXXXX XXXX and was ...,Fraud or scam
2,I have a particular account that is stating th...,Incorrect information on your report
3,I have not supplied proof under the doctrine o...,Attempts to collect debt not owed
4,Hello i'm writing regarding account on my cred...,Incorrect information on your report


In [4]:
complaints['Issue'].value_counts().sort_index()

Issue
Attempts to collect debt not owed        73163
Communication tactics                    21243
Fraud or scam                            12347
Incorrect information on your report    229305
Struggling to pay mortgage               17374
Name: count, dtype: int64

In [5]:
seed = 123
for statement in complaints.loc[complaints['Issue'] == 'Attempts to collect debt not owed', 'Consumer complaint narrative'].sample(3, random_state=seed):
    print(statement)
    print('-----------------------------')

This company in which I hold no contract with nor have received services from reported ( 3 ) different collection accounts against my Social Security Number in the amount of {$510.00}, {$710.00} and {$570.00} with XXXX, XXXX & XXXX   credit reporting agencies. I requested verification and validation on XX/XX/2018 of the alleged debt and account, however, the business failed to provide adequate proof. Considering this business does not have a contract with me for goods or services they have provided nor have they provided adequate proof, I am not obligated to pay for the alleged debt.
-----------------------------
An affidavit of Billing Error Notice was mailed to XXXX XXXX XXXX XXXX and/or XXXX, XXXX but they didn't respond back. 

The account is an agreement, not a contract. Based on the consumer protection laws and your lack of complete disclosure I rescind the entire transaction due to fraud. 

CONSUMER PROTECTION LAWS and U.S. CODE VIOLATIONS : Equal Credit Opportunity Act / Truth 

In [6]:
seed = 123
for statement in complaints.loc[complaints['Issue'] == 'Communication tactics', 'Consumer complaint narrative'].sample(3, random_state=seed):
    print(statement)
    print('-----------------------------')

The company name is Valentine and Kebartas. 
After missimg multiple calls a day from this company I finally spoke with someone on XXXX/XXXX/16. XXXX had sent my final bill to my old address and I never got it. The person I spoke to at Valentine and Kebartas corrected my address and arranged to send out a reprint of the bill. She waved the ridiculous {$5.00} fee to have the bill reprinted. I let her know that I would be taking care of the bill as soon as I received it. 
Not 1 day later the calls started again. 
I received a call this morning by a very pushy caller and was told that if I was taken off the call list without making payment arrangements my bill would go into collections. I asked why my file had n't been updated to show that I was cooperating and s ( he ) said their system just does n't show everything. 
When I complained about their repetitive calls the caller said that legally the system could call my phone up to 6 times per day. This is harrassment and also threatening by

In [7]:
# Preform train test split with default behavior
X = complaints[['Consumer complaint narrative']]
y = complaints['Issue']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 321, stratify = y)

In [20]:
# Create vectorizer object to convert words to zeros and ones for our model
vect = CountVectorizer()

# Fit vectorizer to our train test split
X_train_vec = vect.fit_transform(X_train['Consumer complaint narrative'])
X_test_vec = vect.transform(X_test['Consumer complaint narrative'])

In [9]:
X_train_vec

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 21821225 stored elements and shape (265074, 72222)>

In [41]:
# Fit naive bayes model
nb = MultinomialNB().fit(X_train_vec, y_train)

y_pred = nb.predict(X_test_vec)

In [11]:
# Display confusion matrix and view baesline accuracy
print(f'Accuracy: {accuracy_score(y_test, y_pred)}')
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7988976663120487
[[12086  2039   500  3343   323]
 [  587  4476    51   112    85]
 [   67    55  2813   110    42]
 [ 6610  1046   834 46997  1839]
 [   36    40    12    38  4217]]


In [31]:
# Assemble dataframe with log probabilities for each feature in naive bayes model
coef_df = pd.DataFrame({
    'word': vect.get_feature_names_out(),
    'coef_debt_not owed': nb.feature_log_prob_[0],
    'coef_comm_tactic': nb.feature_log_prob_[1],
    'coef_fraud_scam': nb.feature_log_prob_[2],
    'coef_incorrect_info': nb.feature_log_prob_[3],
    'coef_struggling_mort': nb.feature_log_prob_[4]
})

In [38]:
coef_df.sort_values(by='coef_struggling_mort', ascending=False).head(10)

,word,coef_debt_not owed,coef_comm_tactic,coef_fraud_scam,coef_incorrect_info,coef_struggling_mort
70263,xxxx,-2.910857,-3.107376,-2.871778,-2.171077,-3.065259
62544,the,-3.153222,-3.337110,-3.104480,-3.271571,-3.102004
63746,to,-3.368785,-3.273903,-3.277243,-3.507347,-3.272055
7647,and,-3.559412,-3.461158,-3.521274,-3.683953,-3.534046
70253,xx,-4.354451,-4.414672,-4.300641,-3.906537,-3.997421
42576,my,-3.976358,-4.004805,-3.920115,-3.802165,-4.012266
44819,of,-3.785469,-4.301982,-4.170980,-3.951249,-4.020873
62471,that,-4.110257,-4.195261,-4.174660,-4.281850,-4.170567
68649,was,-4.545667,-4.453072,-4.117623,-4.806341,-4.227363
33986,in,-4.358560,-4.648182,-4.581898,-4.512169,-4.237444


> ### Pre-Processing Error
>
>* We can see here that without a preprocessing step in our pipeline we are left with short words and redacted documents (XXXX). These throw off our model. Because these phrases occur more often in the english language our model assumes they have a great effect on predictions which is simply untrue and driven by **skewed estimates.**

In [27]:
# Create tool to test different words effect on our model
word = 'drugs'

probs = np.exp(nb.feature_log_prob_)[:, vect.vocabulary_[word]]

In [25]:
# View effects of words across multiple categories
for label, prob in zip(nb.classes_, probs):
    print(f"Class: {label:<20} | Probability: {prob:.8f}")

Class: Attempts to collect debt not owed | Probability: 0.00009571
Class: Communication tactics | Probability: 0.00014525
Class: Fraud or scam        | Probability: 0.00011561
Class: Incorrect information on your report | Probability: 0.00009381
Class: Struggling to pay mortgage | Probability: 0.00014658


In [14]:
# Build transformer with preprocessing steps
vect = CountVectorizer()
clf = MultinomialNB()

pipe = Pipeline([("vect", vect), ("clf", clf)])

param_grid = {
    'vect__ngram_range':[(1,1), (1,2), (1,3)],
    'vect__min_df':[1, 2, 5, 10, 20],
    'clf__fit_prior':[False, True]
}

In [23]:
# Utilize randomSearchCV to fine-tune optimal hyper parameters
rs = RandomizedSearchCV(estimator = pipe, param_distributions = param_grid, verbose = 2, n_jobs = -1)
rs.fit(X_train['Consumer complaint narrative'], y_train)

# Save document to files so we don't need to re-run code
dump(rs, "../models/cv_01.joblib")

['../models/cv_01.joblib']

In [ ]:
* What steps did you take to preprocess the data?

### CountVectorizer() Pre-processing Behavior

> In this notebook a lot of the preprocessing is handled by CountVectorize. It automatically lowercases everything and standardizes my text. CountVectorizer() also removes any punctuation and reverts to this typical regular expression **(token_pattern='(?u)\\b\\w\\w+\\b')** pattern to split ong paragraphs into individual tokens. It also removes any special characters like XXXX in redacted documents. CountVectorizer is also set up to ignore any single letter phrases.
>
>* Ignore short phrases like "I" or "a"
>* Parse all phrases into single words (tokens)
>* Vectorize these words into numeric format to generate model

In [ ]:
* How did a count vectorizer compare to a tfidf vectorizer?

### Quantitative Performance (Accuracy)
>
>* CountVectorizer Baseline: ~79.89% accuracy
>* TfidfVectorizer Model: 81.98% accuracy
>
>Switching from a raw frequency count to TF-IDF provided an immediate performance boost of ~2.1%. Count vectorizer or "Bag Of Words" simply counts how many times a word appears in a specific complaint narrative. TfidfVectorizor (Term Frequency-Inverse Document Frequency) penalizes words that show up repeatedly across the entire document. This allows more specific and context rich output.

In [55]:
# Define tfidf preprocessing steps
tfidf_vect = TfidfVectorizer(
    token_pattern=r'\b(?![xX]{2,})\w{3,}\b', 
    stop_words='english',
    ngram_range=(1, 1)    # Adjust for Bi-grams and Tri-grams
)

# Bundle it into a new pipeline
tfidf_pipe = Pipeline([
    ("vect", tfidf_vect), 
    ("clf", MultinomialNB())
])

# Fit the pipeline on your training data
tfidf_pipe.fit(X_train['Consumer complaint narrative'], y_train)

Pipeline(steps=[('vect',
                 TfidfVectorizer(stop_words='english',
                                 token_pattern='\\b(?![xX]{2,})\\w{3,}\\b')),
                ('clf', MultinomialNB())])

### Bi-gram and Tri-gram Evaluation
>
> In the naive bayes model we can see an effect from number of morphemes used in our model. Single words increase overall accuracy of the model. As we increase the size of phrases used to train the model It loses accuracy. We saw descent accuracy with our single gram naive bayes model, it generated predictions with **82% accuracy.** After acounting for bi-grams the accuracy dropped to 66% an then 62% after including tri-gram phrases.
>
> 

In [56]:
# Predict on the test data
y_pred_tfidf = tfidf_pipe.predict(X_test['Consumer complaint narrative'])

# View accuracy of model
print(f"TFIDF Vectorizer Accuracy: {accuracy_score(y_test, y_pred_tfidf):.4f}")

TFIDF Vectorizer Accuracy: 0.8198


In [57]:
# Extract the trained TFIDF vectorizer and Naive Bayes classifier from the pipeline
trained_tfidf_vect = tfidf_pipe.named_steps['vect']
trained_tfidf_clf = tfidf_pipe.named_steps['clf']

# Build the DataFrame using the components extracted from tfidf_pipe
tfidf_coef_df = pd.DataFrame({
    'word': trained_tfidf_vect.get_feature_names_out(),
    'coef_debt_not owed': trained_tfidf_clf.feature_log_prob_[0],
    'coef_comm_tactic': trained_tfidf_clf.feature_log_prob_[1],
    'coef_fraud_scam': trained_tfidf_clf.feature_log_prob_[2],
    'coef_incorrect_info': trained_tfidf_clf.feature_log_prob_[3],
    'coef_struggling_mort': trained_tfidf_clf.feature_log_prob_[4]
})

# View the top words for mortgage complaints to test it out
tfidf_coef_df.sort_values(by='coef_struggling_mort', ascending=False).head(10)

,word,coef_debt_not owed,coef_comm_tactic,coef_fraud_scam,coef_incorrect_info,coef_struggling_mort
41478,mortgage,-8.003996,-8.576088,-9.468474,-6.677527,-4.782138
41112,modification,-10.642352,-10.556096,-10.888250,-8.824527,-4.987499
38536,loan,-6.734526,-7.029844,-8.078943,-5.931293,-4.996738
31803,home,-7.243411,-7.152499,-8.078629,-7.037633,-5.430407
28123,foreclosure,-9.512400,-10.315065,-10.671313,-9.035212,-5.549837
46354,payments,-7.282178,-6.982120,-7.719119,-5.948150,-5.633446
46332,payment,-6.341364,-6.132943,-6.744634,-5.670925,-5.693824
27980,forbearance,-11.081031,-10.118813,-10.880682,-7.633208,-5.848786
55054,sale,-8.567055,-10.057498,-8.483595,-9.071457,-5.883458
62946,told,-6.117087,-5.507441,-6.166005,-6.534021,-5.897796


In [58]:
# View the top words for debt not owed to test it out
tfidf_coef_df.sort_values(by='coef_debt_not owed', ascending=False).head(10)

,word,coef_debt_not owed,coef_comm_tactic,coef_fraud_scam,coef_incorrect_info,coef_struggling_mort
19519,debt,-4.270572,-5.036296,-8.494294,-5.979785,-7.952305
4918,account,-4.717973,-6.443454,-5.494041,-4.534205,-7.144216
18413,credit,-4.783204,-6.501547,-7.288615,-4.255980,-7.864095
15549,collection,-4.934075,-6.119414,-9.561130,-6.258282,-9.158047
16077,company,-5.026177,-5.494027,-6.799917,-5.880738,-6.391561
52732,report,-5.230288,-7.398007,-7.550027,-4.540864,-8.763038
34365,information,-5.463329,-6.472725,-7.115738,-4.789292,-7.117705
45634,owe,-5.545796,-6.486135,-9.313102,-7.291039,-8.079857
45793,paid,-5.552451,-6.904715,-7.231223,-5.876062,-7.144768
50915,received,-5.614714,-6.008688,-6.479431,-6.240755,-6.240054


In [59]:
# View the top words for incorrect info to test it out
tfidf_coef_df.sort_values(by='coef_incorrect_info', ascending=False).head(10)

,word,coef_debt_not owed,coef_comm_tactic,coef_fraud_scam,coef_incorrect_info,coef_struggling_mort
18413,credit,-4.783204,-6.501547,-7.288615,-4.255980,-7.864095
4918,account,-4.717973,-6.443454,-5.494041,-4.534205,-7.144216
52732,report,-5.230288,-7.398007,-7.550027,-4.540864,-8.763038
34365,information,-5.463329,-6.472725,-7.115738,-4.789292,-7.117705
52777,reporting,-5.696252,-8.187396,-9.043439,-4.847581,-9.269064
4972,accounts,-6.101066,-8.122170,-7.590713,-4.889300,-9.769683
10169,balance,-6.220676,-7.649254,-7.879869,-5.118437,-7.624814
32668,identity,-5.869046,-8.686038,-8.155034,-5.260837,-10.457795
61820,theft,-5.902408,-9.279962,-8.506199,-5.309728,-10.480855
17069,consumer,-5.924357,-7.479214,-8.422929,-5.339514,-8.531794


### Key words Influencing Classification
>
> Above we have five different complaint categories and in each dataframe we can view the impact of different word on our models prediction. Because our model is relatively accurate we can see which words contribute to our model pooling these into different categories accurately. We can see in fraud and scam categories words like **"paypal" and "coinbase"** often implicate fraud.

In [60]:
# View the top words for fraud scam to test it out
tfidf_coef_df.sort_values(by='coef_fraud_scam', ascending=False).head(10)

,word,coef_debt_not owed,coef_comm_tactic,coef_fraud_scam,coef_incorrect_info,coef_struggling_mort
41234,money,-6.496844,-6.889813,-5.032011,-6.932765,-7.013229
10286,bank,-6.519908,-7.065954,-5.227039,-6.310472,-6.178065
4918,account,-4.717973,-6.443454,-5.494041,-4.534205,-7.144216
46410,paypal,-7.828092,-9.017194,-5.658867,-9.922089,-11.983174
29132,funds,-8.335301,-8.840067,-5.969694,-8.946347,-7.662823
63475,transfer,-7.701237,-8.971415,-5.990652,-8.980375,-8.138102
56127,sent,-5.669662,-6.649969,-6.051064,-5.895447,-6.380430
24017,email,-6.960209,-7.163646,-6.057397,-7.385202,-7.118516
15402,coinbase,-12.826702,-11.929399,-6.125273,-13.806231,-11.983174
55018,said,-6.319754,-6.108753,-6.131668,-6.823810,-6.473276


In [61]:
# View the top words for communication tactics to test it out
tfidf_coef_df.sort_values(by='coef_comm_tactic', ascending=False).head(10)

,word,coef_debt_not owed,coef_comm_tactic,coef_fraud_scam,coef_incorrect_info,coef_struggling_mort
13062,calls,-6.705712,-4.890127,-8.125420,-7.607843,-7.528968
13042,calling,-7.017887,-4.928585,-8.128908,-8.489193,-8.064497
47189,phone,-6.145961,-5.002480,-6.440774,-6.886154,-7.044280
19519,debt,-4.270572,-5.036296,-8.494294,-5.979785,-7.952305
13015,called,-6.009020,-5.251822,-6.392948,-6.548112,-6.410702
43620,number,-6.031893,-5.318190,-6.356426,-5.981238,-7.588321
19334,day,-7.115714,-5.390846,-7.051038,-7.037029,-7.340509
16077,company,-5.026177,-5.494027,-6.799917,-5.880738,-6.391561
62690,times,-6.500846,-5.494029,-7.560634,-6.420960,-6.925234
59310,stop,-6.959814,-5.502690,-7.639519,-8.256973,-7.797513


### Models Evaluated
>
> Throughout this analysis, we tracked performance using overall accuracy across 5 core complaint categories. The baseline naive bayes model has 79.89% accuracy. It is incredibly fast to train and establish a solid prediction floor. this model is easily distracted by structural data artifacts (like xxxx) commonly used in redacted documents or public data.
>
>* #### Unigram Model (TfidfVectorizer + MultinomialNB): 81.98% accuracy
> 
>This was our most successful configuration. Applying a TF-IDF vectorizer and filtering out both stop words and character noise allowed the Naive Bayes model to prioritize distinct financial domain terms, boosting accuracy by over 2%.
>
>* #### N-gram Scale Models
>
>These model we're useful to explore multi phrase passages and thier effect on classification. The models suffered a severe drop in performance. Expanding the feature space to pairs and triplets created massive vocabulary sparsity that overwhelmed the counting mechanics of the Multinomial Naive Bayes algorithm.

**Bonus:** A larger dataset containing 20 additional categories can be downloaded from https://drive.google.com/file/d/1gW6LScUL-Z7mH6gUZn-1aNzm4p4CvtpL/view?usp=share_link. How well do your models work with these additional categories?